# Luna vs. cheaper agent models

Run the Week 1 Day 2 claims agent with Luna and two lower-priced models, then compare accuracy, cost, and speed.


In [78]:
import json
import os
import time
from pathlib import Path
from typing import Literal
import pandas as pd
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

In [79]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [80]:
DATA = Path("../../data")
def load(name):
    return json.loads((DATA / name).read_text(encoding="utf-8"))
claims = load("claims.json")
labels = {x["claim_id"]: x for x in load("evals/decisions.json")}
policies = {x["policy_id"]: x for x in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")
print(f"{len(claims)} claims")

25 claims


In [81]:
recorded = {}


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

In [82]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the claim using the policy wording and customer policy. Approve covered claims with the payout from section 10; decline uncovered claims; refer missing or inconsistent information."""
AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer policy and record your decision.

{policy_wording}
"""
def make_claim_prompt(claim):
    return "# Claim\n\n" + json.dumps(claim, indent=2)
def is_right(answer, label):
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1

In [ ]:
def run_agent_candidate(model, effort=None):
    """Run the same claims agent across the queue with the selected model."""
    kwargs = {"reasoning_effort": effort} if effort else {}
    llm = ChatOpenAI(model=model, base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, **kwargs)
    agent = create_agent(model=llm, tools=[get_policy, record_decision], system_prompt=AGENT_SYSTEM_PROMPT)
    results=[]; recorded.clear()
    for claim in claims:
        start=time.time()
        result=agent.invoke({"messages":[{"role":"user","content":make_claim_prompt(claim)}]})
        answer=recorded.get(claim["claim_id"],{"decision":"nothing recorded","payout":0.0,"rationale":""})
        label=labels[claim["claim_id"]]
        cost=sum(m.response_metadata["token_usage"]["cost"] for m in result["messages"] if m.type=="ai")
        results.append({"model":model,"effort":effort or "default","claim_id":claim["claim_id"],"right":is_right(answer,label),"cost":cost,"seconds":round(time.time()-start,1)})
        time.sleep(2)  # Agent uses multiple requests per claim; throttle below new-account RPM limit.
    return results

## Run and compare

Each run processes the full queue and may incur API charges. The pause helps avoid OpenRouter request limits.

In [88]:
# openai/gpt-6-luna
luna_results = run_agent_candidate("openai/gpt-6-luna", "low")
print(f"openai/gpt-6-luna: {sum(r['right'] for r in luna_results)} / {len(luna_results)} correct")

openai/gpt-6-luna: 17 / 25 correct


In [86]:
# meta/muse-ember
muse_ember_results = run_agent_candidate("meta/muse-ember", "low")
print(f"meta/muse-ember: {sum(r['right'] for r in muse_ember_results)} / {len(muse_ember_results)} correct")

meta/muse-ember: 18 / 25 correct


In [87]:
# anthropic/claude-haiku-5.5
haiku_results = run_agent_candidate("anthropic/claude-haiku-5.5", "low")
print(f"anthropic/claude-haiku-5.5: {sum(r['right'] for r in haiku_results)} / {len(haiku_results)} correct")

anthropic/claude-haiku-5.5: 16 / 25 correct


In [89]:
all_results = luna_results + muse_ember_results + haiku_results
summary = pd.DataFrame(all_results).groupby(["model", "effort"], as_index=False).agg(correct=("right", "sum"), claims=("right", "count"), total_cost=("cost", "sum"), total_seconds=("seconds", "sum"))
summary["cost_per_claim"] = summary["total_cost"] / summary["claims"]
summary["seconds_per_claim"] = summary["total_seconds"] / summary["claims"]
summary

,model,effort,correct,claims,total_cost,total_seconds,cost_per_claim,seconds_per_claim
0,anthropic/claude-haiku-5.5,low,16,25,0.050606,188.8,0.002024,7.552
1,meta/muse-ember,low,18,25,0.030784,139.8,0.001231,5.592
2,openai/gpt-6-luna,low,17,25,0.007418,227.0,0.000297,9.080
